In [ ]:
import json
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
OUTPUT = TASK/'output'
prep = TASK/'input/fit/data'
config=json.loads((ROOT/'config.json').read_text())
models=config['primary_plot_models']
panel=pd.read_csv(prep/'panel.csv',index_col=0).to_numpy().T
emp=panel[:,np.isfinite(panel).all(axis=0)]
labels=dict(zip(models,['GRW','AR(1)','H-AR(1)','HD-H']))
valid=np.isfinite(panel[:-1]) & np.isfinite(panel[1:]) & (panel[:-1]>0)
edges=np.unique(np.quantile(panel[:-1][valid],np.linspace(0,1,11)))[1:-1]
drop=pd.read_csv(OUTPUT/'dropout_by_productivity.csv').query("source=='empirical_available' and stage=='pooled'").sort_values('bin')
reentry=pd.read_csv(OUTPUT/'reentry_errors.csv').query("source=='ar1_s'").sort_values('duration')
durations=reentry.duration.to_numpy()
DRAWS = 200
Y=config['last_year']

In [ ]:
def measure(q):
 t=q[-4:].sum(axis=0);s=np.sort(t);n=len(s)
 qq=stats.probplot(np.log2(t[t>0]))[1][2]**2
 gini=2*np.dot(np.arange(1,n+1),s)/(n*s.sum())-(n+1)/n
 ranks=stats.rankdata(q,axis=1)
 ranks-=ranks.mean(axis=1,keepdims=True)
 ranks/=np.linalg.norm(ranks,axis=1,keepdims=True)
 fixed=ranks[1:]@ranks[0]
 lag=np.einsum('ij,ij->i',ranks[5:],ranks[:-5])
 active=q[:-1]>0
 bins=np.searchsorted(edges,q[:-1][active],side='left')
 nbin=np.bincount(bins,minlength=len(drop))
 zeros=np.bincount(bins,weights=(q[1:][active]==0),minlength=len(drop))
 dropout=np.sqrt(np.average((np.divide(zeros,nbin,out=np.full(len(nbin),np.nan),where=nbin>0)-drop.probability)**2,weights=drop.n))
 risk=np.zeros(Y+2);restart=risk.copy();run=np.zeros(q.shape[1],dtype=int)
 for t0 in range(Y):
  run=np.where(q[t0]==0,run+1,0)
  risk+=np.bincount(run,minlength=len(risk))
  restart+=np.bincount(run,weights=q[t0+1]>0,minlength=len(risk))
 hazard=np.divide(restart[durations],risk[durations],out=np.full(len(durations),np.nan),where=risk[durations]>0)
 err=np.sqrt(np.average((hazard-reentry.empirical)**2,weights=reentry.n_empirical_risk))
 keys=['FourYearQQRtwo','FourYearMedianTotal','FourYearMeanTotal','FourYearSDTotal','FourYearGini','AnnualWassersteinMean','RankInitialFinal','RankInitialFourYearSlope','RankLagFiveFinal','RankLagFiveFourYearSlope','DropoutRMSEPoints','RestartHazardRMSEPoints']
 return dict(zip(keys,[qq,np.median(t),np.mean(t),np.std(t),gini,np.mean([stats.wasserstein_distance(a,b) for a,b in zip(emp,q)]),fixed[-1],stats.linregress(np.arange(1,Y+1),fixed).slope,lag[-1],stats.linregress(np.arange(5,Y+1),lag).slope,dropout*100,err*100]))

In [ ]:
# replay the existing simulators with fixed fits and alternate RNG seeds
rows = []
with tempfile.TemporaryDirectory() as tmp:
 out=Path(tmp)/'out';out.mkdir()
 fit=Path(tmp)/'fit';fit.mkdir()
 (fit/'data').symlink_to(prep)
 for model in models:
  (fit/model).symlink_to(ROOT/'c_fit/output'/model)
  p,=(ROOT/'d_simulate/src/manuscript').glob(f'*_{model}.ipynb')
  s='\n'.join(''.join(cell['source']) for cell in json.loads(p.read_text())['cells']).split('np.save(')[0]
  start=s.index('TASK ='); end=s.index('EPS, Y, SEED',start)
  s=s[:start]+"TASK = ROOT/'d_simulate'\nINPUT = FIT\nOUTPUT = OUT\nCONFIG = {**BASE_CONFIG, 'seed': RUN_SEED}\n"+s[end:]
  code=compile(s,str(p),'exec')
  for seed in range(DRAWS):
   scope=dict(ROOT=ROOT,FIT=fit,OUT=out,BASE_CONFIG=config,RUN_SEED=seed)
   exec(code,scope)
   values=measure(scope['sim'])
   rows.append(dict(seed=seed,model=labels[model],**values))

In [ ]:
values = pd.DataFrame(rows).set_index(['seed','model'])
empirical = pd.Series(measure(emp))
loss = values.sub(empirical).abs()
for metric in ['AnnualWassersteinMean','DropoutRMSEPoints','RestartHazardRMSEPoints']:
    loss[metric] = values[metric]
qq = 'FourYearQQRtwo'
signed_qq = values[qq] - empirical[qq]
loss = loss.drop(columns=qq)
reentry_metric = 'RestartHazardRMSEPoints'
# compare common metrics within each seed and check two missing-reentry conventions
losses = {
    'common':loss.mask(loss.isna().groupby(level='seed').transform('any')),
    'missing_last':loss.fillna(np.inf),
    'without_reentry':loss.drop(columns=reentry_metric),
}
results = []
for rule, errors in losses.items():
    mean_rank = errors.groupby(level='seed').rank().mean(axis=1)
    place = mean_rank.unstack().rank(axis=1,method='min').stack().reindex(values.index)
    results.append(pd.DataFrame({'mean_rank':mean_rank,'place':place,
                                 'metrics_compared':errors.notna().sum(axis=1),
                                 'qq_signed_difference':signed_qq}).assign(rule=rule))
rankings = pd.concat(results)
per_run = values.join(results[0].drop(columns='rule'))
per_run.to_csv(OUTPUT/'seed_comparison_runs.csv')
rankings.to_csv(OUTPUT/'seed_comparison_rankings.csv')
summary = (rankings.assign(first=rankings.place.eq(1), second=rankings.place.eq(2),
                           qq_below_dblp=rankings.qq_signed_difference.lt(0))
           .groupby(['rule','model'])[['first','second','qq_below_dblp']].sum().reset_index())
summary.to_csv(OUTPUT/'seed_comparison_summary.csv',index=False)
display(summary)